In [2]:
# decorator


#- Think of a decorator as a wrapper around a function.

def my_decorator(func):
    def wrapper():
        print("Before function call")
        func()
        print("After function call")
    return wrapper

@my_decorator
@my_decorator
def say_hello():
    print("Hello!")

say_hello()

Before function call
Before function call
Hello!
After function call
After function call


In [ ]:
# generator
 - A generator in Python is a special type of function that produces values one at a time using yield, instead of returning all values at once.

def count_up_to(n):
    for i in range(1, n + 1):
        yield i

gen = count_up_to(3)

print(next(gen))  # 1
print(next(gen))  # 2
print(next(gen))  # 3     


# Iterator
class Counter:
    def __init__(self, n):
        self.n = n
        self.i = 0

    def __iter__(self):
        return self

    def __next__(self):
        if self.i < self.n:
            val = self.i
            self.i += 1
            return val
        raise StopIteration

In [ ]:
## Mutable vs Immutable
Mutable → can change after creation (list, dict, set)
lst1 = [1, 2]
lst2 = lst1
lst1.append(3)
print(lst2)  # [1, 2, 3] (same object modified)

Immutable → cannot change (new object created instead) (int, float, str, tuple)
x = 10
y = x
x = 20
print(y)  # 10 (new object created)

# Default mutable arguments persist across calls.
def func(a=[]):
    a.append(1)
    return a

print(func())  # [1]
print(func())  # [1, 1] 



## Shallow vs deep
import copy
lst1 = [[1, 2], [3, 4]]

shallow = copy.copy(lst1)
deep = copy.deepcopy(lst1)

lst1[0][0] = 100
print(shallow)  # [[100, 2], [3, 4]] ❗
print(deep)     # [[1, 2], [3, 4]] ✅



## *args vs **kwargs
*args → variable positional arguments (tuple)
**kwargs → variable keyword arguments (dict)
def func(*args, **kwargs):
    print(args)
    print(kwargs)

func(1, 2, 3, name="Sourav", age=25)

# Output
    (1, 2, 3)
    {'name': 'Sourav', 'age': 25}




In [1]:
# GIL
Python is slower mainly because it is an interpreted, dynamically typed language with additional runtime overhead like the GIL
GIL (Global Interpreter Lock) = Mutex that allows only ONE thread to execute Python bytecode at a time, limiting CPU-bound parallelism.

# Multiple Inheritance & MRO
Python supports multiple inheritance and resolves method calls using MRO (Method Resolution Order).
class A:
    def show(self):
        print("A")

class B(A):
    def show(self):
        print("B")

class C(A):
    def show(self):
        print("C")

class D(B, C):
    pass

d = D()
d.show() # B

print(D.mro()) # [D, B, C, A, object]


# .super() Always use super() instead of calling parent directly
class B(A):
    def show(self):
        super().show()
    

[1]
[1, 1]


In [ ]:
# Dunder Methods: Dunder (magic) methods define object behavior
class User:
    def __init__(self, name):
        self.name = name

    def __str__(self):
        return f"User: {self.name}"

u = User("Sourav")
print(u) # User: Sourav


def __repr__(self):
    return f"User('{self.name}')"
print(repr(u))

In [ ]:
# Thread and Locks

import threading

lock = threading.Lock()
counter = 0

def increment():
    global counter
    for _ in range(100000):
        with lock:
            counter += 1

t1 = threading.Thread(target=increment)
t2 = threading.Thread(target=increment)

t1.start()
t2.start()

t1.join() # It blocks the main thread until thread t1 finishes execution
t2.join()

print(counter)

In [ ]:
# Django
Django (Django) is a high-level Python web framework for rapid development

- MTV architecture (Model-Template-View)
    Model    → Database layer
    Template → UI layer
    View     → Business logic

- ORM (no raw SQL needed)
- Built-in admin panel
- Security (CSRF, XSS, SQL injection protection)


# Middleware
👉 Middleware = layer between request & response
- usecase:
    - Authentication
    - Logging
    - Rate limiting
    - Caching
    - Request/response modification

class SimpleMiddleware:
    def __init__(self, get_response):
        self.get_response = get_response

    def __call__(self, request):
        print("Before view")

        response = self.get_response(request)

        print("After view")
        return response


# Sync vs Async View
# Sync
def my_view(request):
    return HttpResponse("Hello")

# Async
async def my_view(request):
    return HttpResponse("Hello")
- External API calls
- DB calls (async-supported)
- IO-bound tasks

# WSGI vs ASGI
WSGI → sync only
ASGI → supports async + websockets

In [ ]:
# @staticmethod @classmethod
Type            | First Param | Access to Class | Access to Instance | Use Case
----------------|-------------|-----------------|--------------------|-------------------------
Instance Method | self        | ✅               | ✅                  | Work with object data
Class Method    | cls         | ✅               | ❌                  | Work with class-level logic, It does NOT allocate memory
Static Method   | none        | ❌               | ❌                  | Utility/helper functions

class Demo:
    value = 10

    @classmethod
    def class_method(cls):
        return cls.value

    @staticmethod
    def static_method():
        return "No access to class"

In [ ]:
# ETL
 - Data Handling: Extract → Transform → Load

# PR Review keypoints in Django

In [ ]:
1. DB constraints
    Same employee can be assigned to same project multiple times

class Meta:
    constraints = [
        models.UniqueConstraint(
            fields=["employee", "project"],
            name="unique_employee_project"
        )
    ]


2. Check if data is cleaned before adding it to DB
3. Add DB index
class Meta:
    indexes = [
        models.Index(fields=["employee"]),
    ]

3.1 Foreign Key query:
    # Wrong code
    for emp in employees:
        print(emp.department.name)

    # Correct code
    employees = EmployeeProfile.objects.select_related("department")
    for emp in employees:
        print(emp.department.name)

    
4. Serializer conditions
from django.db.models import Sum
total = employee.assignments.aggregate(
    total=Sum("allocation_percentage")
)["total"] or 0


5. exclude
employee.assignments.exclude(id=self.instance.id)

6. Check Views
    authentication_classes = [JWTAuthentication]
    permission_classes = [IsAuthenticated]

7. View return valid status code
8. Check proper logging
9. Check authentication
10. test cases
11. Multiple statement in transaction block
    from django.db import transaction
    
    with transaction.atomic():
        serializer.save()


In [ ]:
# Django DB

from django.db import transaction

with transaction.atomic():
    accounts = Account.objects.select_for_update().filter(id__in=[a_id, b_id]).order_by("id")

    sender = ...
    receiver = ...

    if sender.balance < amount:
        raise Exception("Insufficient balance")

    sender.balance -= amount
    receiver.balance += amount

    sender.save()
    receiver.save()